# 3. Snowflake: import the Databricks Ossie

This notebook reads the Ossie file that Databricks produced and creates a semantic
view from it. The measure added in Databricks (`TOTAL_QUANTITY`) comes across.

The import is a single built-in function, `SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML`,
the mirror of the native export in notebook 1. Snowflake reads and writes Ossie
natively, so this side needs no converter and no shim. The Databricks side used the
open-source Apache converter because Databricks has no native Ossie functions yet.

Read this before running: importing under a name replaces any existing semantic view
of that name. The incoming file names its model `SALES_SV_V2`, so by default this
creates a separate view and leaves your original `SALES_SV` untouched. Step 3 lets
you choose the target name.

Set `DATABASE` and `SCHEMA` to the same names you used in notebook 1 and Databricks.
The table references inside the Ossie file point at that database and schema, so they
have to match for the imported view to resolve. No admin role is required.

## Step 1 - Set your database and schema

In [ ]:
# Set the database and schema to work in. Pick a database and schema where your
# current role can create tables, a stage, and a semantic view. No admin role is
# needed; the notebook runs with your role's privileges. Use these same names in
# the Databricks notebook so the table references in the Ossie file line up.
DATABASE = "DEMOS"
SCHEMA   = "SEMANTIC_INTEROP"
print(f"Working in {DATABASE}.{SCHEMA}")

In [ ]:
USE SCHEMA {{DATABASE}}.{{SCHEMA}};

## Step 2 - Upload the Databricks file, then read it

Upload `ossie_from_databricks.yaml` to the stage (Snowsight stage browser, or
`snow stage copy ossie_from_databricks.yaml @<database>.<schema>.INTEROP_STAGE`). The
file format reads the whole document as one value.

In [ ]:
CREATE FILE FORMAT IF NOT EXISTS {{DATABASE}}.{{SCHEMA}}.RAW_TEXT_FMT
  TYPE = 'CSV' FIELD_DELIMITER = NONE RECORD_DELIMITER = NONE ESCAPE_UNENCLOSED_FIELD = NONE;

In [ ]:
LIST @{{DATABASE}}.{{SCHEMA}}.INTEROP_STAGE;

In [ ]:
SET yaml_content = (
  SELECT $1 FROM @{{DATABASE}}.{{SCHEMA}}.INTEROP_STAGE/ossie_from_databricks.yaml
  (FILE_FORMAT => '{{DATABASE}}.{{SCHEMA}}.RAW_TEXT_FMT')
);

In [ ]:
SELECT $yaml_content;

## Step 3 - Choose the target view name

Set `target_view` to control what gets created. This matters when you reuse the demo:

- Keep both views (default): leave it as `SALES_SV_V2`. Your original `SALES_SV` stays
  as is, and the imported view lands beside it.
- Replace the original: set it to `SALES_SV`. The import overwrites `SALES_SV` with the
  round-tripped definition.
- Keep a history: set it to `SALES_SV_V3`, `SALES_SV_V4`, and so on for each run.

The next cell rewrites the model name in the file to `target_view` before import. The
check after it returns a row if a view of that name already exists, so you know if you
are about to replace one.

In [ ]:
SET target_view = 'SALES_SV_V2';

In [ ]:
SET yaml_to_import = (SELECT REPLACE($yaml_content, 'SALES_SV_V2', $target_view));

In [ ]:
SHOW SEMANTIC VIEWS IN SCHEMA {{DATABASE}}.{{SCHEMA}};
-- A row here means a view of your target name already exists and will be replaced.
SELECT "name" AS existing_view_with_target_name
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "name" = $target_view;

## Step 4 - Import

`SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML` turns the Ossie file straight into a
semantic view. Snowflake does this natively, so there is nothing to convert here and
no shim to run; it is the same native path as the export in notebook 1, in reverse.

The call creates, or replaces if the name exists, the semantic view named by
`target_view`.

In [ ]:
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML('{{DATABASE}}.{{SCHEMA}}', $yaml_to_import);

## Step 5 - Verify

The round-tripped view returns the same numbers as Databricks: EAST 12/750/5, WEST 11/700/5.

In [ ]:
SET target_fqn = '{{DATABASE}}.{{SCHEMA}}.' || $target_view;
SELECT * FROM SEMANTIC_VIEW(
  IDENTIFIER($target_fqn)
  DIMENSIONS region
  METRICS total_quantity, total_order_amount, order_count
) ORDER BY region;